# Rede Neural Multicamadas - Análise de Crédito com Keras

Atividade de Inteligência Artificial - 7° período Sistemas de Informação

**Objetivo:** Construir e comparar 3 redes neurais multicamadas com Keras/TensorFlow para classificação de aprovação de crédito.

**Documentação de referência para salvar pesos:** https://keras.io/api/models/model_saving_apis/weights_saving_and_loading/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import tensorflow as tf
from tensorflow import keras
from keras import layers

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow versão: {tf.__version__}")
print(f"Keras versão: {keras.__version__}")

## 1. Carregamento e Exploração dos Dados

In [ ]:
# Carregando o dataset de crédito
df = pd.read_csv('credito.csv')

# Exibindo as primeiras linhas para entender a estrutura
display(df.head())

# Verificando as informações das colunas
print("\n--- Informações do Dataset ---")
df.info()

print(f"\nFormato do dataset: {df.shape[0]} registros, {df.shape[1]} colunas")

In [ ]:
# Verificando estatísticas descritivas
display(df.describe())

# Verificando a distribuição da variável alvo
print("\n--- Distribuição da variável alvo (Approved) ---")
print(df['Approved'].value_counts())
print(f"\nProporção: {df['Approved'].value_counts(normalize=True).to_dict()}")

## 2. Pré-processamento dos Dados

In [ ]:
# Removendo a coluna ZipCode (não contribui para a classificação)
if 'ZipCode' in df.columns:
    df = df.drop('ZipCode', axis=1)

# Codificando colunas categóricas com LabelEncoder
colunas_categoricas = ['Industry', 'Ethnicity', 'Citizen']
le = LabelEncoder()
for col in colunas_categoricas:
    df[col] = le.fit_transform(df[col].astype(str))

print("Dados após codificação das variáveis categóricas:")
display(df.head())

In [ ]:
# Separando features (X) e variável alvo (y)
X = df.drop('Approved', axis=1)
y = df['Approved']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nColunas das features: {list(X.columns)}")

## 3. Divisão dos Dados (Treino, Validação e Teste)

In [ ]:
# Primeira divisão: 80% treino+validação, 20% teste
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Segunda divisão: dos 80%, separar 75% treino e 25% validação (resulta em 60/20/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Normalização dos dados com StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Conjunto de Treino:     {X_train_scaled.shape[0]} amostras ({X_train_scaled.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de Validação:  {X_val_scaled.shape[0]} amostras ({X_val_scaled.shape[0]/len(X)*100:.0f}%)")
print(f"Conjunto de Teste:      {X_test_scaled.shape[0]} amostras ({X_test_scaled.shape[0]/len(X)*100:.0f}%)")
print(f"\nNúmero de features: {X_train_scaled.shape[1]}")

## 4. Definição das 3 Redes Neurais

| Rede | Camadas Ocultas | Neurônios | Ativação | Diferencial |
|------|-----------------|-----------|----------|-------------|
| **Rede 1** | 2 | 64, 32 | ReLU | Baseline simples |
| **Rede 2** | 3 | 128, 64, 32 | ReLU + Dropout | Mais profunda com regularização |
| **Rede 3** | 3 | 64, 32, 16 | tanh, ReLU, ReLU | Funções de ativação mistas |

In [ ]:
# Número de features de entrada
n_features = X_train_scaled.shape[1]

def criar_rede_1():
    """Rede 1: Baseline simples com 2 camadas ocultas (ReLU)"""
    modelo = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(n_features,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

def criar_rede_2():
    """Rede 2: Mais profunda com 3 camadas ocultas (ReLU + Dropout)"""
    modelo = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(n_features,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

def criar_rede_3():
    """Rede 3: Funções de ativação mistas (tanh + ReLU)"""
    modelo = keras.Sequential([
        layers.Dense(64, activation='tanh', input_shape=(n_features,)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

print("Funções de criação das 3 redes definidas com sucesso!")

## 5. Treinamento com Early Stopping

O treinamento deve parar se, em mais de **20 épocas**, o erro (loss de validação) não diminuir.

In [ ]:
# Callback de Early Stopping (parar se em 20 épocas o erro não diminuir)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

# --- Treinamento da Rede 1 ---
print("=" * 60)
print("TREINAMENTO DA REDE 1 - Baseline Simples (64, 32 - ReLU)")
print("=" * 60)

modelo_1 = criar_rede_1()
historico_1 = modelo_1.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# --- Treinamento da Rede 2 ---
print("=" * 60)
print("TREINAMENTO DA REDE 2 - Profunda com Dropout (128, 64, 32)")
print("=" * 60)

modelo_2 = criar_rede_2()
historico_2 = modelo_2.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# --- Treinamento da Rede 3 ---
print("=" * 60)
print("TREINAMENTO DA REDE 3 - Ativações Mistas (tanh + ReLU)")
print("=" * 60)

modelo_3 = criar_rede_3()
historico_3 = modelo_3.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping],
    verbose=1
)

## 6. Visualização do Treinamento

In [ ]:
# Gráficos de Loss e Accuracy para as 3 redes
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.suptitle('Histórico de Treinamento das 3 Redes Neurais', fontsize=16, fontweight='bold')

historicos = [historico_1, historico_2, historico_3]
nomes = ['Rede 1 (64, 32 - ReLU)', 'Rede 2 (128, 64, 32 - ReLU + Dropout)', 'Rede 3 (64, 32, 16 - tanh/ReLU)']

for i, (hist, nome) in enumerate(zip(historicos, nomes)):
    # Gráfico de Loss
    axes[i, 0].plot(hist.history['loss'], label='Treino', linewidth=2)
    axes[i, 0].plot(hist.history['val_loss'], label='Validação', linewidth=2)
    axes[i, 0].set_title(f'{nome} - Loss')
    axes[i, 0].set_xlabel('Época')
    axes[i, 0].set_ylabel('Loss')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    # Gráfico de Accuracy
    axes[i, 1].plot(hist.history['accuracy'], label='Treino', linewidth=2)
    axes[i, 1].plot(hist.history['val_accuracy'], label='Validação', linewidth=2)
    axes[i, 1].set_title(f'{nome} - Acurácia')
    axes[i, 1].set_xlabel('Época')
    axes[i, 1].set_ylabel('Acurácia')
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Avaliação das 3 Redes (Acurácia, Precisão e Recall)

In [ ]:
# Função para avaliar um modelo
def avaliar_modelo(modelo, X_test, y_test, nome):
    """Avalia o modelo e retorna as métricas"""
    # Fazendo predições
    y_pred_prob = modelo.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()

    # Calculando métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    print(f"\n{'=' * 60}")
    print(f"Relatório - {nome}")
    print(f"{'=' * 60}")
    print(classification_report(y_test, y_pred, target_names=['Não Aprovado (0)', 'Aprovado (1)']))

    return {'Modelo': nome, 'Acurácia': acc, 'Precisão': prec, 'Recall': rec, 'Predições': y_pred}

# Avaliando as 3 redes
modelos = [modelo_1, modelo_2, modelo_3]
nomes_modelos = [
    'Rede 1 (64, 32 - ReLU)',
    'Rede 2 (128, 64, 32 - Dropout)',
    'Rede 3 (64, 32, 16 - tanh/ReLU)'
]

resultados = []
for modelo, nome in zip(modelos, nomes_modelos):
    resultado = avaliar_modelo(modelo, X_test_scaled, y_test, nome)
    resultados.append(resultado)

In [ ]:
# Tabela comparativa das métricas
df_resultados = pd.DataFrame([{
    'Modelo': r['Modelo'],
    'Acurácia': f"{r['Acurácia']:.4f}",
    'Precisão': f"{r['Precisão']:.4f}",
    'Recall': f"{r['Recall']:.4f}"
} for r in resultados])

print("\n" + "=" * 60)
print("TABELA COMPARATIVA DAS 3 REDES NEURAIS")
print("=" * 60)
display(df_resultados)

# Identificando a rede com maior recall
recalls = [r['Recall'] for r in resultados]
idx_melhor = np.argmax(recalls)
print(f"\n>>> A rede com MAIOR RECALL é: {resultados[idx_melhor]['Modelo']}")
print(f"    Recall = {resultados[idx_melhor]['Recall']:.4f}")

## 8. Parâmetros da Rede com Maior Recall

In [ ]:
# Selecionando o melhor modelo (maior recall)
melhor_modelo = modelos[idx_melhor]
melhor_nome = nomes_modelos[idx_melhor]
melhor_historico = historicos[idx_melhor]

print(f"{'=' * 60}")
print(f"MELHOR MODELO: {melhor_nome}")
print(f"{'=' * 60}")
print(f"\nAcurácia: {resultados[idx_melhor]['Acurácia']:.4f}")
print(f"Precisão: {resultados[idx_melhor]['Precisão']:.4f}")
print(f"Recall:   {resultados[idx_melhor]['Recall']:.4f}")
print(f"Épocas treinadas: {len(melhor_historico.history['loss'])}")
print(f"\n--- Arquitetura do Modelo ---")
melhor_modelo.summary()

## 9. Salvando os Pesos do Melhor Modelo

Conforme a documentação: https://keras.io/api/models/model_saving_apis/weights_saving_and_loading/

In [ ]:
# Salvando os pesos do melhor modelo como artefato
# Referência: https://keras.io/api/models/model_saving_apis/weights_saving_and_loading/
caminho_pesos = 'melhor_modelo_pesos.weights.h5'
melhor_modelo.save_weights(caminho_pesos)

print(f"Pesos do melhor modelo salvos com sucesso em: '{caminho_pesos}'")
print(f"\nPara carregar os pesos novamente:")
print(f"  modelo.load_weights('{caminho_pesos}')")

## 10. Tabela Comparativa - Classes Reais vs Predições do Melhor Modelo

Gerando uma amostra de teste para comparar as classes reais com as respostas do melhor modelo, justificando o valor do recall.

In [ ]:
# Predições do melhor modelo no conjunto de teste
y_pred_prob = melhor_modelo.predict(X_test_scaled, verbose=0)
y_pred_final = (y_pred_prob > 0.5).astype(int).flatten()
y_test_array = y_test.values

# Criando a tabela comparativa
df_comparacao = pd.DataFrame({
    'Classe Real': y_test_array,
    'Predição do Modelo': y_pred_final,
    'Acertou': ['✅ Sim' if real == pred else '❌ Não' for real, pred in zip(y_test_array, y_pred_final)]
})

print(f"{'=' * 60}")
print(f"TABELA COMPARATIVA - {melhor_nome}")
print(f"{'=' * 60}")
print(f"\nAmostra de teste ({len(df_comparacao)} registros):")
display(df_comparacao)

# Resumo dos acertos e erros
total = len(df_comparacao)
acertos = (y_test_array == y_pred_final).sum()
erros = total - acertos

print(f"\n--- Resumo ---")
print(f"Total de amostras: {total}")
print(f"Acertos: {acertos} ({acertos/total*100:.1f}%)")
print(f"Erros: {erros} ({erros/total*100:.1f}%)")

In [ ]:
# Análise detalhada do Recall
# Recall = TP / (TP + FN) - mede quantos positivos reais foram identificados corretamente

# Filtrando apenas os casos que são realmente positivos (Approved = 1)
positivos_reais = df_comparacao[df_comparacao['Classe Real'] == 1]
positivos_acertados = positivos_reais[positivos_reais['Predição do Modelo'] == 1]
positivos_errados = positivos_reais[positivos_reais['Predição do Modelo'] == 0]

print(f"{'=' * 60}")
print(f"JUSTIFICATIVA DO RECALL")
print(f"{'=' * 60}")
print(f"\nRecall = TP / (TP + FN)")
print(f"\nTotal de créditos realmente aprovados (positivos reais): {len(positivos_reais)}")
print(f"Modelo identificou corretamente (TP): {len(positivos_acertados)}")
print(f"Modelo errou (FN - falsos negativos): {len(positivos_errados)}")
print(f"\nRecall = {len(positivos_acertados)} / ({len(positivos_acertados)} + {len(positivos_errados)}) = {len(positivos_acertados)/len(positivos_reais):.4f}")

print(f"\n--- Casos em que o modelo ERROU (Falsos Negativos) ---")
print(f"Esses são créditos que DEVERIAM ter sido aprovados, mas o modelo classificou como não aprovados:")
if len(positivos_errados) > 0:
    display(positivos_errados)
else:
    print("Nenhum falso negativo! O modelo identificou todos os positivos corretamente.")